In [ ]:
# 01 Dataset exploration
#
# Question. How are scores distributed overall and per frozen partition?
#
# Type. Exploratory, labels only (official test labels are not loaded). Prerequisite. notebook 00 passes.
from pathlib import Path
import os

REPO = Path.cwd()
while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
    REPO = REPO.parent
os.chdir(REPO)

import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ct_iqa.data.splits import load_manifest


def show(title, obj):
    """Print a section title, then the table as plain text."""
    print(f"\n=== {title} ===")
    print(obj.to_string())   # plain text table, no interactive viewer

labels = json.loads(Path("data/labels/ldct_iqac/train.json").read_text())
manifest = load_manifest("experiments/study_v2/protocol/split_manifest.json", labels)
parts = {p: np.array([labels[n] for n in manifest[p]]) for p in ("train", "validation", "calibration")}


In [ ]:
show("Score statistics per partition (raw 0-4 scale)", pd.DataFrame({p: pd.Series(v).describe() for p, v in parts.items()}).T.round(3))

fig, axes = plt.subplots(1, 3, figsize=(12, 3.4), sharey=True)
bins = np.arange(-0.1, 4.3, 0.2)
for ax, (p, v) in zip(axes, parts.items()):
    ax.hist(v, bins=bins)
    ax.axvline(v.mean(), color="k", ls="--", label=f"mean {v.mean():.2f}")
    ax.set(title=f"{p} (n={len(v)})", xlabel="score (0-4)")
    ax.legend()
axes[0].set_ylabel("images")
fig.tight_layout()
plt.show()


In [ ]:
# Optional: example images
# Reads a few training images; off by default.
SHOW_IMAGES = False
if SHOW_IMAGES:
    from PIL import Image

    names = sorted(labels, key=labels.get)[::250][:4]
    fig, axes = plt.subplots(1, len(names), figsize=(12, 3))
    for ax, n in zip(axes, names):
        ax.imshow(np.array(Image.open(f"data/raw/ldct_iqac/train/image/{n}")), cmap="gray")
        ax.set_title(f"{n}: {labels[n]:.1f}")
        ax.axis("off")
    plt.show()
